# Probability, conditional probability and base rates

Core, Part 1 (Statistics I), lesson C4. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** compute the probability of an event and of two independent events, read P(A given B) and P(B given A) as different questions, update a belief with Bayes using a table of 1,000, and explain base-rate neglect with an example.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it.

## Setup

In [ ]:
import math
import numbers

import linecache
import sys
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
EXERCISES = ["Exercise 1", "Exercise 2", "Exercise 3"]


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong):
            print("Hint:", message)
    return False

def show_error(code, short=False):
    """Run a few lines that fail on purpose and print the error, so Run all keeps going."""
    code = code.strip("\n") + "\n"
    linecache.cache["<cell>"] = (len(code), None, code.splitlines(True), "<cell>")
    try:
        exec(compile(code, "<cell>", "exec"), globals())
    except Exception as error:
        if short:
            print("".join(traceback.format_exception_only(type(error), error)).strip().split("\n")[-1])
        else:
            traceback.print_exception(type(error), error, error.__traceback__.tb_next, file=sys.stdout)

# One look for every chart of the lesson: small figure, no top and right frame, no legend frame, space made automatically.
BLUE, MAGENTA, GOLD, GREEN = "#0072b2", "#d6479a", "#ad9100", "#00a07a"  # the colors of the site's figures
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[BLUE, MAGENTA, GOLD, GREEN]),
    "figure.figsize": (4.5, 2.8),
    "figure.constrained_layout.use": True,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")
# Orders of customers with a loyalty card, with the tier of the customer: 1,799 of the 1,800 orders (the walk-in order has no customer).
known = orders.dropna(subset=["customer_id"]).merge(customers[["customer_id", "loyalty_tier"]], on="customer_id", validate="many_to_one")
assert len(known) == 1799
print("`orders` has 1,800 rows; `known` has the 1,799 orders that belong to a customer, with `loyalty_tier`.")

## Unit 1: Probability as a share, and independence

**Predict first.** the share of orders paid by card is about 0.56 and the share taken at Old Town is about 0.41. If the payment method and the store told you nothing about each other, what share of all orders would be card orders at Old Town? Will the real share be close?

In [ ]:
card = orders["payment_method"] == "card"
old_town = orders["store_id"] == 1
print(round(card.mean(), 3), round(old_town.mean(), 3))
print(round(card.mean() * old_town.mean(), 3), round((card & old_town).mean(), 3))

## Unit 2: Conditional probability, read both ways

**Predict first.** among the orders of customers, what share of the Harbor orders belong to gold customers, and what share of the gold customers' orders were taken at Harbor? Which share is larger?

In [ ]:
gold = known["loyalty_tier"] == "gold"
harbor = known["store_id"] == 3
both = int((gold & harbor).sum())
print(both, int(harbor.sum()), int(gold.sum()))
print(round(both / harbor.sum(), 3), round(both / gold.sum(), 3))

## Unit 3: Bayes with a table of 1,000

**Predict first.** a made-up check flags orders that may hold a keying mistake. Suppose 2 orders in 100 hold one, the check flags 9 in 10 of the mistakes, and it also flags 5 in 100 of the correct orders. Of 1,000 orders, how many are flagged, and what share of the flagged orders are real mistakes? Guess before you run: about 9 in 10, about half, or about 1 in 4?

In [ ]:
n = 1000
mistakes = 20  # made up: 2 in 100 orders hold a keying mistake
caught = 18  # made up: the check flags 9 in 10 of them
false_alarms = 49  # made up: and 5 in 100 of the 980 correct orders
table = pd.DataFrame({"flagged": [caught, false_alarms], "not flagged": [mistakes - caught, n - mistakes - false_alarms]}, index=["mistake", "correct"])
print(table)
print(caught + false_alarms, round(caught / (caught + false_alarms), 3))

**Predict first.** Bayes' theorem relabels the same table: P(mistake given flag) = P(flag given mistake) × P(mistake) ÷ P(flag). Will it give the same share as the table? And will it give back the share of Harbor orders that belong to gold customers from unit 2, from the other direction?

In [ ]:
p_mistake, p_flag_given_mistake, p_flag_given_correct = 0.02, 0.90, 0.05
p_flag = p_flag_given_mistake * p_mistake + p_flag_given_correct * (1 - p_mistake)
print(round(p_flag_given_mistake * p_mistake / p_flag, 3))
p_harbor_given_gold, p_gold, p_harbor = both / gold.sum(), gold.mean(), harbor.mean()
print(round(p_harbor_given_gold * p_gold / p_harbor, 3))

## Unit 4: Base-rate neglect

**Predict first.** the check never changes: it flags 9 in 10 mistakes and 5 in 100 correct orders. Only how common the mistakes are changes. What share of the flagged orders are real mistakes when 2 in 100 orders hold one, when 20 in 100 do, and at the real rate of the data, one line in 3,287 with an extra zero?

In [ ]:
def real_share(base_rate, hit=0.90, false_alarm=0.05):
    return hit * base_rate / (hit * base_rate + false_alarm * (1 - base_rate))

print(round(real_share(0.02), 3), round(real_share(0.20), 3), round(real_share(1 / len(order_items)), 4))

## Finish it

### Exercise 1: Finish it

**Task:** What share of the Old Town orders, store 1, were paid by mobile? Use all 1,800 rows of `orders`. Put the share in `answer_1` as a number between 0 and 1, rounded to 3 decimals.

**Expected:** `answer_1` is a number between 0 and 1.

In [ ]:
old_town_orders = orders[orders["store_id"] == 1]  # given: the Old Town orders
# find the share of them paid by mobile, and set answer_1
answer_1 = None  # replace None

In [ ]:
_ = check("Exercise 1", "answer_1", 0.298, traps={0.397: "That is the share of mobile orders that were taken at Old Town, the other direction. The question starts from Old Town."})

<details>
<summary>Hint</summary>

The share is out of the Old Town orders, so the denominator is the number of Old Town orders.

</details>

<details>
<summary>Solution</summary>

```python
old_town_orders = orders[orders["store_id"] == 1]
answer_1 = round(float((old_town_orders["payment_method"] == "mobile").mean()), 3)
```

217 of the 729 Old Town orders, 0.298. The other direction, the share of the 546 mobile orders that were taken at Old Town, is 0.397, a different question with a different denominator; the check cell reports it.

</details>

## On your own

### Exercise 2: On your own

**Task:** A made-up check on loyalty cards. Suppose 5 cards in 100 are copies. The check flags 8 in 10 of the copies and 1 in 10 of the genuine cards. Before you start, say in one sentence what the result looks like: a count and a share. Build the table for 1,000 cards, then put in `answer_2` the number of flagged cards and the share of the flagged cards that are copies, rounded to 3 decimals, as a list in that order. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a list of a whole number and a share.

In [ ]:
_ = check("Exercise 2", "answer_2", [135, 0.296], traps={(135, 0.8): "0.8 is how many copies the check flags, P(flag given copy). The question asks which share of the flagged cards are copies, P(copy given flag)."})

<details>
<summary>Hint</summary>

Start from 1,000 cards: how many are copies, how many genuine, and how many of each group are flagged?

</details>

<details>
<summary>Solution</summary>

```python
cards = 1000
copies = 50
genuine = cards - copies
copies_flagged = 40
genuine_flagged = 95
flagged = copies_flagged + genuine_flagged
answer_2 = [flagged, round(copies_flagged / flagged, 3)]
```

50 copies and 950 genuine cards; the check flags 40 copies and 95 genuine ones, 135 in all, and 40 of 135 is 0.296. Only about 3 flagged cards in 10 are copies, although the check catches 8 copies in 10. The check cell reports 0.8, the share of copies that are flagged.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague predicts the number of orders by gold customers taken at Campus, store 2, by multiplying the share of gold orders by the share of Campus orders, among the 1,799 orders of `known`, and by 1,799. Count the real orders by gold customers at Campus and set `answer_3` to the colleague's prediction, rounded to a whole number, and the real count, in that order.

**Expected:** `answer_3` is a list of two whole numbers.

In [ ]:
gold_share = (known["loyalty_tier"] == "gold").mean()
campus_share = (known["store_id"] == 2).mean()
predicted = round(gold_share * campus_share * len(known))  # the colleague's prediction
real = predicted  # replace with the real count, which the colleague never looked at
answer_3 = [predicted, real]

In [ ]:
_ = check("Exercise 3", "answer_3", [69, 31], traps={(69, 69): "The second number must be the real count of gold orders at Campus, not the prediction again."})

<details>
<summary>Hint</summary>

Filter `known` on both conditions and count the rows.

</details>

<details>
<summary>Solution</summary>

```python
gold_share = (known["loyalty_tier"] == "gold").mean()
campus_share = (known["store_id"] == 2).mean()
predicted = round(gold_share * campus_share * len(known))
real = int(((known["loyalty_tier"] == "gold") & (known["store_id"] == 2)).sum())
answer_3 = [predicted, real]
```

The multiplication rule predicts 69 orders, and the data holds 31. Multiplying the two shares is right only when the two events are independent; here the gold customers' orders are not spread over the stores like all orders, so the prediction is more than twice the real count. The check cell reports the prediction twice.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")